# NHL Home Game Predictor — Feature Engineering

Build rolling window features, opponent features, and derived features.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pipeline.data_loader import load_all_seasons
from pipeline import features, noise_reduction

df_raw = load_all_seasons()
print('Raw shape:', df_raw.shape)

In [ ]:
# Full feature engineering
df_fe = features.build_features(df_raw)
print('After feature engineering:', df_fe.shape)
df_fe.head(3)

In [ ]:
# Inspect rolling feature columns
rolling_cols = [c for c in df_fe.columns if c.startswith('rolling_')]
print(f'Rolling feature count: {len(rolling_cols)}')
rolling_cols[:20]

In [ ]:
# NaN rates per rolling feature
nan_rates = df_fe[rolling_cols].isnull().mean().sort_values(ascending=False)
print(nan_rates[nan_rates > 0].head(20))

In [ ]:
# Apply noise reduction
df_clean = noise_reduction.apply(df_fe.copy())
print('Noise reduction applied. New cols:', [c for c in df_clean.columns if 'extreme' in c])

In [ ]:
# Rolling win pct over time for one team
team = df_clean[df_clean['team'] == 'TOR'].sort_values('date')
plt.figure(figsize=(12, 4))
for w in [5, 10, 20]:
    col = f'rolling_{w}_win_pct'
    if col in team.columns:
        plt.plot(team['date'], team[col], label=f'N={w}')
plt.title('TOR Rolling Win % Over Time')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Correlation of rolling features with outcome
df_clean['win_binary'] = df_clean['result'].eq('W').astype(int)
feat_cols = features.get_feature_cols(df_clean)
corr_with_target = df_clean[feat_cols + ['win_binary']].corr()['win_binary'].drop('win_binary')
corr_with_target.sort_values().plot(kind='barh', figsize=(8, 14), title='Feature Correlation with Win')
plt.tight_layout()
plt.show()